# AI-Powered Fake News Detection Using Text Classification
**Institution:** Indian Institute of Computing and Technology (IICT)

### Project Objectives:
1. **Preprocessing:** Clean raw news articles using custom punctuation removal, lowercasing, and lemmatization.
2. **Feature Extraction:** Build an n-gram TF-IDF pipeline using up to 5,000 features.
3. **Model Development:** Train and evaluate KNN, Logistic Regression, Random Forest, and a Neural Network.
4. **Visualizations:** Generate ROC curves, Confusion Matrices, and explore keyword coefficients.

## Step 1: Import Dependencies

In [ ]:
import os
import re
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, classification_report

# NLTK Initialization
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Setup completed successfully.")

## Step 2: Load dataset

In [ ]:
data_path = "../data/raw/fake_or_real_news.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/fake_or_real_news.csv"

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['label'].value_counts())
display(df.head())

### Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='Oranges')
plt.title('Real vs. Fake News Counts')
plt.grid(axis='y', alpha=0.3)
plt.show()

## Step 3: Preprocessing and Text Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_news_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', '', text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(cleaned)

df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
df['label_num'] = df['label'].map({'FAKE': 1, 'REAL': 0})

print("Cleaning text...")
df['cleaned_text'] = (df['title'] + " " + df['text']).apply(clean_news_text)
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)
print("Data cleaning finished!")

## Step 4: Feature Extraction (TF-IDF)

In [ ]:
X = df['cleaned_text']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"Train shape: {X_train_tfidf.shape}, Test shape: {X_test_tfidf.shape}")

## Step 5: Model Building and Tuning

In [ ]:
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Logistic Regression': LogisticRegression(C=10.0, max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42, early_stopping=True)
}

results = {}
y_preds = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_tfidf, y_train)
    
    y_pred = model.predict(X_test_tfidf)
    y_preds[name] = y_pred
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0)
    }
    print(f"{name} metrics calculated.")

## Step 6: Model Comparison Table

In [ ]:
results_df = pd.DataFrame(results).T
display(results_df)

## Step 7: Performance Visualizations

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_tfidf)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(y_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[idx], cbar=False,
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'])
    axes[idx].set_title(f'Confusion Matrix: {name}')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
lr_model = models['Logistic Regression']
coefs = lr_model.coef_[0]
features = vectorizer.get_feature_names_out()

top_indices = coefs.argsort()[::-1][:20]
top_features = [features[i] for i in top_indices]
top_coefs = [coefs[i] for i in top_indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_coefs, y=top_features, hue=top_features, palette='rocket', legend=False)
plt.title('Top 20 Indicative Words of Fake News (Logistic Regression Coefficients)')
plt.xlabel('Coefficient Value')
plt.ylabel('Word')
plt.tight_layout()
plt.show()